[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syriascitech/Medad-CV-Bootcamp/blob/main/Week8/week_8_object_tracking.ipynb)


<style>
.rtl-cell {
    direction: rtl !important;
    text-align: right !important;
    line-height: 1.9 !important;
    font-size: 17px !important;
    font-family: Arial, Tahoma, sans-serif !important;
}
.rtl-cell p,
.rtl-cell h1, .rtl-cell h2, .rtl-cell h3, .rtl-cell h4,
.rtl-cell li, .rtl-cell blockquote, .rtl-cell th, .rtl-cell td {
    direction: rtl !important;
    text-align: right !important;
}
.rtl-cell ul, .rtl-cell ol {
    direction: rtl !important;
    text-align: right !important;
    padding-right: 2em !important;
    padding-left: 0 !important;
}
.rtl-cell blockquote {
    border-right: 4px solid #d0d7de !important;
    border-left: none !important;
    margin-right: 0 !important;
    padding-right: 1em !important;
}
.rtl-cell table {
    direction: rtl !important;
    text-align: right !important;
    margin-right: 0 !important;
    margin-left: auto !important;
    border-collapse: collapse !important;
}
.rtl-cell th, .rtl-cell td { padding: 7px 10px !important; }
.rtl-cell code { direction: ltr !important; unicode-bidi: isolate !important; }
.rtl-cell pre, .rtl-cell pre code {
    direction: ltr !important;
    text-align: left !important;
    unicode-bidi: embed !important;
}
.rtl-cell .todo {
    background-color: rgba(249, 168, 37, 0.14);
    border-right: 5px solid #F9A825;
    padding: 12px 18px;
    margin: 16px 0;
    border-radius: 6px;
}
.rtl-cell .note {
    background-color: rgba(127, 127, 127, 0.12);
    border-right: 5px solid #777;
    padding: 12px 18px;
    margin: 16px 0;
    border-radius: 6px;
}
</style>

<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h1>الأسبوع 8: تتبع الأجسام</h1>
<h2>Object Tracking</h2>
<hr />
<p><strong>مسار أنظمة المرور الذكية – الرؤية الحاسوبية</strong></p>
<p><strong>المُعد/المؤلف:</strong> المهندس عامر صوان والمهندس حسن صوان</p>
<hr />

<p>في الأسبوع الماضي درّبنا نموذج YOLO على التعرّف على سيارات الإسعاف إلى
جانب السيارات والشاحنات والحافلات. حصلنا على نموذج يجيب بثقة عن سؤالين في
كل إطار على حدة: <strong>ما هذا؟</strong> و<strong>أين هو؟</strong></p>

<p>لكن جرّب هذا السؤال: كم مركبة عبرت التقاطع في آخر دقيقة؟ النموذج وحده لا
يستطيع الإجابة، لأنه <strong>ينسى كل شيء بين إطار وإطار</strong>. لا يعرف أن
سيارة الإسعاف التي رآها في الإطار العاشر هي نفسها التي رآها في الإطار
التاسع. كل إطار بالنسبة له عالم جديد كلياً.</p>

<p>هذا الأسبوع نحلّ هذه المشكلة بـ<strong>التتبع Object Tracking</strong>:
نعطي كل جسم رقماً ثابتاً يلازمه عبر الإطارات، فنستطيع أخيراً الإجابة عن
أسئلة مثل "كم مركبة عبرت؟" و"هل هذا الإسعاف يقترب أم يبتعد؟" - وهذا بالضبط
ما يحتاجه الأسبوع التاسع لبناء منطق أولوية الإشارة.</p>

<div class="note">
<p><strong>كيف يُقرأ هذا الدفتر:</strong> أُعدّ هذا الدفتر وشُغِّل بالكامل
على Google Colab مسبقاً، والمخرجات المحفوظة أمامك هي نتائج حقيقية لا
توقُّعات. في الحصة نفتحه ونقرأه معاً دون تشغيل أي خلية. إن أردت تشغيله
بنفسك في البيت، اضغط شارة Colab أعلى الدفتر وشغّله من الأعلى للأسفل على
جهاز جديد كلياً.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>أهداف الدرس</h2>

<p>في نهاية هذا الدرس ستكون قادراً على أن:</p>

<ol>
<li>تشرح لماذا لا يكفي الكشف وحده للإجابة عن أسئلة تراكمية مثل العدّ.</li>
<li>تفرّق بين <strong>Detection</strong> و <strong>Track</strong> و
<strong>ID</strong>، وتشرح فلسفة Tracking-by-Detection.</li>
<li>تحسب <strong>IoU</strong> بين صندوقين وتبني مصفوفة تكلفة، وتفهم متى
تفشل المطابقة الجشعة (Greedy Matching).</li>
<li>تشرح بحدسٍ سليم كيف يتنبأ <strong>مرشّح كالمان</strong> بموضع جسم أثناء
الحجب Occlusion.</li>
<li>تقارن بين SORT و DeepSORT و ByteTrack: ماذا يضيف كل واحد، وبأي ثمن.</li>
<li>تشغّل <code>model.track()</code> في Ultralytics وتقرأ معرّفات
التتبع من نتائجه.</li>
<li>تبني متتبعاً مبسّطاً بأيدينا وتشاهد <strong>تبديل هوية ID Switch</strong>
يحدث أمامك، فتفهم لماذا نحتاج ما هو أذكى من المطابقة الجشعة.</li>
<li>تشخّص أعطال التتبع الشائعة: تبديل الهوية، التجزّؤ، والمسارات الوهمية.</li>
</ol>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>محتويات الدرس</h2>

<ol>
<li>لماذا الكشف وحده لا يكفي؟</li>
<li>تعريف المشكلة: التتبع متعدد الأجسام Multi-Object Tracking</li>
<li>الربط بين الإطارات Data Association</li>
<li>التنبؤ بالحركة: مرشّح كالمان Kalman Filter</li>
<li>SORT</li>
<li>DeepSORT: إضافة المظهر</li>
<li>ByteTrack: الفكرة البسيطة الذكية</li>
<li>عملياً مع Ultralytics</li>
<li>نبني متتبع IoU مصغّراً من الصفر</li>
<li>رسم المسارات Trails وتلوين حسب الـ ID</li>
<li>تشغيل على المقطع الكامل وحفظ الناتج</li>
<li>لمحة أولى: كيف يفتح التتبع باب العدّ؟</li>
<li>متى يفشل المتتبع؟</li>
<li>كيف نقيس جودة التتبع؟</li>
<li>تمرين صفي</li>
<li>أسئلة مراجعة سريعة</li>
<li>بنك أسئلة Kahoot</li>
<li>الخلاصة</li>
</ol>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>التهيئة</h1>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>هذا الدفتر يعمل على <strong>Google Colab</strong>: يحتاج إنترنت
لتثبيت المكتبات ولاستنساخ ملفات هذا الدرس مرة واحدة عند بداية كل جلسة
Colab جديدة. الخلية التالية تثبّت المكتبات بصمت.</p>

</div>


In [ ]:
import sys

!{sys.executable} -m pip install -q ultralytics opencv-python pandas matplotlib pyyaml numpy scipy imageio-ffmpeg
!{sys.executable} -m pip install -q torch torchvision


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>جلسة Colab الجديدة لا تملك أياً من ملفات هذا الدرس: الفيديوهات
والنموذج المدرَّب والأشكال. الخلية التالية تستنسخ المستودع مرة واحدة فقط،
ولا تفعل شيئاً إن كانت الملفات موجودة أصلاً (كأن تشغّل الدفتر محلياً من
داخل مجلد <code>Week8</code>).</p>

</div>


In [ ]:
import os
from pathlib import Path

# على Colab لا توجد ملفات المستودع، فننسخها مرة واحدة.
if not Path("videos").exists() and not Path("models/emergency_best.pt").exists():
    !git clone -q https://github.com/syriascitech/Medad-CV-Bootcamp.git /content/repo
    os.chdir("/content/repo/Week8")

print("مجلد العمل:", os.getcwd())


In [ ]:
import importlib.util

REQUIRED = ["ultralytics", "cv2", "torch", "pandas", "matplotlib", "yaml", "scipy"]

missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]

if missing:
    print("المكتبات الناقصة:", "، ".join(missing))
    print("أعد تشغيل الخلية الأولى في هذا القسم، ثم أعد تشغيل النواة.")
else:
    print("كل المكتبات المطلوبة متوفرة.")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>الاستيرادات وإعدادات العمل</h2>

<p>مكتبة Ultralytics تحاول الاتصال بالإنترنت في أكثر من موضع حتى أثناء
الاستدلال: لتحميل الأوزان إن لم تجدها محلياً، ولإرسال إحصاءات استخدام.
الخلية التالية تشير إلى نسخة النموذج المحفوظة محلياً وتُسكِت هذه المحاولات.
كما تضبط دقة الرسوم البيانية لإبقاء حجم هذا الدفتر معقولاً رغم أنه يُحفظ
بكل مخرجاته.</p>

</div>


In [ ]:
import os

# يمنع تعارض OpenMP بين Ultralytics و OpenCV على بعض الأجهزة.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["YOLO_VERBOSE"] = "False"

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import linear_sum_assignment

from ultralytics import YOLO, settings

settings.update({"sync": False})  # إيقاف إرسال إحصاءات الاستخدام

# دقة معتدلة لكل الرسوم البيانية، فلا يتضخّم حجم الدفتر مع عشرات المخرجات.
plt.rcParams["figure.dpi"] = 90

MODEL_PATH = "models/emergency_best.pt"  # مسار محلي: لا تحميل من الإنترنت

print("PyTorch:", torch.__version__)
print("النموذج:", MODEL_PATH)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>الجهاز المستخدم</h2>

<p>على Colab غالباً يتوفّر كرت شاشة T4 مجاني. إن كنت تشغّل هذا الدفتر
محلياً على معالج فقط، ستعمل كل الخلايا لكنها ستستغرق وقتاً أطول - القيم
القصوى للإطارات في هذا الدرس مضبوطة لتبقى معقولة على المعالج أيضاً.</p>

</div>


In [ ]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("الجهاز المستخدم:", DEVICE)
print("عدد أنوية المعالج المتاحة:", os.cpu_count())


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>تحميل النموذج المدرَّب من الأسبوع السابع</h2>

<p>هذا هو النموذج نفسه الذي درَّبناه الأسبوع الماضي على أربع فئات، ونسخناه
هنا في <code>Week8/models/</code> ليعمل هذا الدرس بشكل مستقل. لن ندرّب أي
شيء اليوم - سنستخدم هذا النموذج كما هو، ونضيف عليه القدرة على التتبع.</p>

</div>


In [ ]:
model = YOLO(MODEL_PATH)

print("الفئات التي يعرفها النموذج:", model.names)
print("عدد الفئات:", len(model.names))


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>1. لماذا الكشف وحده لا يكفي؟</h1>

<h2>1.1 تجربة حية: ثلاثة إطارات متتالية</h2>

<p>لنبدأ بتجربة بسيطة. المجلد <code>frames/</code> يحوي ثلاثة إطارات
متتالية مستخرجة من <code>videos/traffic_clip.mp4</code>، بفارق جزء من
الثانية بين كل إطار والذي يليه. سنشغّل نموذجنا على كل إطار على حدة، تماماً
كما فعلنا الأسبوع الماضي.</p>

</div>


In [ ]:
from pathlib import Path

frame_paths = sorted(Path("frames").glob("*.jpg"))
print("عدد الإطارات:", len(frame_paths))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, frame_path in zip(axes, frame_paths):
    result = model.predict(source=str(frame_path), conf=0.25, device=DEVICE, verbose=False)[0]
    detected_cars = sum(1 for c in result.boxes.cls if model.names[int(c)] == "car")

    ax.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{frame_path.name}\ncars detected: {detected_cars}", fontsize=12)
    ax.axis("off")

plt.suptitle("Same detector, three consecutive frames", fontsize=15)
plt.tight_layout()
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>النتيجة: كل إطار من الثلاثة فيه <strong>سيارتان مكتشفتان</strong>.
لكن انظر جيداً إلى الصور: هاتان السيارتان متوقفتان عند إشارة حمراء، ولم
تتحركا عملياً بين الإطارات الثلاثة. النموذج لا "يعرف" هذه الحقيقة - فهو
يعالج كل إطار من الصفر، وكأنه يرى الصورة للمرة الأولى في كل مرة.</p>

<h2>1.2 مشكلة العدّ المضاعف</h2>

<p>لنأخذ هذه الملاحظة إلى أقصاها. لو أردنا الإجابة عن سؤال بسيط: "كم سيارة
ظهرت في هذا المقطع؟"، وكانت طريقتنا الساذجة هي: <strong>اجمع عدد السيارات
المكتشفة في كل إطار على حدة</strong>. لنجرّب هذا فعلياً على المقطع كاملاً:</p>

</div>


In [ ]:
naive_car_count = 0
n_frames_processed = 0

for result in model.predict(
    source="videos/traffic_clip.mp4", conf=0.25, device=DEVICE, stream=True, verbose=False
):
    n_frames_processed += 1
    naive_car_count += sum(1 for c in result.boxes.cls if model.names[int(c)] == "car")

print(f"عدد الإطارات في المقطع: {n_frames_processed}")
print(f"مجموع اكتشافات (car) عبر كل الإطارات: {naive_car_count}")
print(f"لو صدّقنا هذا الرقم حرفياً، لقلنا إن {naive_car_count} سيارة مختلفة عبرت في {n_frames_processed / 30:.0f} ثانية فقط.")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<div class="note">
<p><strong>هذا رقم سخيف، وهذا بالضبط بيت القصيد.</strong> المقطع خمس عشرة
ثانية فقط، وفيه على الأرجح أقل من عشر سيارات مختلفة إجمالاً. المجموع
الكبير الذي حصلنا عليه لا يقيس عدد السيارات، بل يقيس <strong>عدد
الإطارات التي ظهرت فيها سيارة</strong> - وهذان سؤالان مختلفان تماماً.</p>
<p>السيارتان اللتان رأيناهما متوقفتين في 1.1 يُعاد عدّهما في كل واحد من
الإطارات الأربعمئة وتسعة وأربعين، لأن الكشف عديم الذاكرة: لا يعرف أن
السيارة في الإطار رقم 200 هي نفسها التي في الإطار رقم 201.</p>
</div>

<h2>1.3 الكشف مقابل التتبع</h2>

<table>
<thead>
<tr><th></th><th>Detection (كشف)</th><th>Tracking (تتبع)</th></tr>
</thead>
<tbody>
<tr><td>يُجيب عن</td><td>ما هذا؟ وأين هو؟ (في إطار واحد)</td>
    <td>هل هذا هو نفسه الذي رأيته سابقاً؟</td></tr>
<tr><td>الذاكرة</td><td>لا ذاكرة بين الإطارات</td>
    <td>يحمل هوية (ID) ثابتة عبر الزمن</td></tr>
<tr><td>يمكّن من</td><td>"يوجد إسعاف في هذا الإطار"</td>
    <td>"هذا الإسعاف نفسه اقترب 30 متراً خلال ثانيتين"</td></tr>
<tr><td>يحتاج</td><td>نموذجاً مدرَّباً فقط</td>
    <td>كاشفاً + خوارزمية ربط بين الإطارات</td></tr>
</tbody>
</table>

<p>التتبّع لا يستبدل الكشف، بل <strong>يبني فوقه</strong>: في كل إطار
نكشف الأجسام كما تعلّمنا، ثم نضيف طبقة جديدة تربط اكتشافات هذا الإطار
باكتشافات الإطار السابق.</p>

<h2>فكر قبل المتابعة</h2>

<ol>
<li>لو حرّكنا الكاميرا بدل السيارة، هل تبقى مشكلة العدّ المضاعف قائمة؟</li>
<li>ما أبسط معلومة يمكن استخدامها للربط بين صندوق في الإطار الحالي وصندوق
في الإطار السابق؟</li>
<li>هل يحتاج التتبع نموذج كشف مختلفاً عن الذي درّبناه الأسبوع الماضي؟</li>
</ol>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>2. تعريف المشكلة: التتبع متعدد الأجسام Multi-Object Tracking</h1>

<h2>2.1 ثلاثة مصطلحات أساسية</h2>

<ul>
<li><strong>Detection (كشف)</strong>: صندوق واحد + فئة + ثقة، ناتج عن
النموذج في إطار واحد فقط. لا يحمل أي معلومة عن الزمن.</li>
<li><strong>Track (مسار)</strong>: سلسلة من الاكتشافات عبر إطارات متتالية،
تنتمي جميعها - حسب اعتقاد المتتبع - لنفس الجسم الفعلي.</li>
<li><strong>ID (هوية)</strong>: رقم صحيح ثابت يُلصَق بمسار واحد طوال حياته.
هو الشيء الوحيد الجديد الذي يضيفه التتبع فوق الكشف.</li>
</ul>

<h2>2.2 فلسفة Tracking-by-Detection</h2>

<p>كل المتتبعات التي سندرسها اليوم (SORT و DeepSORT و ByteTrack) تتبع
الفلسفة نفسها، وهي الأشيع عملياً في الصناعة:</p>

<blockquote>
<p>شغّل كاشفاً جيداً في كل إطار على حدة، ثم اربط اكتشافات الإطار الحالي
بمسارات موجودة من الإطارات السابقة.</p>
</blockquote>

<p>بعبارة أخرى: <strong>لا يوجد نموذج واحد "يتتبّع"</strong>. يوجد كاشف
(النموذج الذي درّبناه) وخوارزمية ربط منفصلة تعمل فوقه. هذا الفصل بين
المهمتين هو ما يجعل هذه الطريقة عملية: يمكن تحسين الكاشف والمتتبع كلٌّ على
حدة.</p>

<h2>2.3 أربعة تحديات تواجه أي متتبع</h2>

<table>
<thead>
<tr><th>التحدي</th><th>ماذا يحدث</th><th>مثال من مقاطعنا</th></tr>
</thead>
<tbody>
<tr>
  <td><strong>الحجب Occlusion</strong></td>
  <td>يختفي الجسم مؤقتاً خلف جسم آخر، فلا يكشفه النموذج لعدة إطارات</td>
  <td>سيارة تختفي خلف حافلة عند التقاطع</td>
</tr>
<tr>
  <td><strong>التشابه Similarity</strong></td>
  <td>جسمان متشابهان جداً بصرياً يتبادلان الأماكن قرب بعضهما</td>
  <td>سيارتان بيضاوان متجاورتان في زحمة السير</td>
</tr>
<tr>
  <td><strong>الدخول والخروج Entry/Exit</strong></td>
  <td>يجب أن يعرف المتتبع متى يولد مسار جديد ومتى يُغلق مسار قديم</td>
  <td>سيارة تدخل من حافة الإطار، وإسعاف يغادر المشهد</td>
</tr>
<tr>
  <td><strong>الحركة السريعة Fast Motion</strong></td>
  <td>يتحرك الجسم مسافة كبيرة بين إطارين فتقل نسبة التداخل بينهما</td>
  <td>إسعاف يتجاوز بسرعة أثناء انخفاض معدل الإطارات</td>
</tr>
</tbody>
</table>

<p>ستلاحظ أن كل تحدٍّ من هذه الأربعة سيظهر لنا عملياً لاحقاً في هذا الدرس -
ليست تحديات نظرية فقط.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>3. الربط بين الإطارات Data Association</h1>

<h2>3.1 IoU كمقياس تشابه</h2>

<p>أبسط فكرة للربط بين صندوق في الإطار الحالي وصندوق في الإطار السابق:
افترض أن الجسم لا يتحرك كثيراً بين إطارين متتاليين (فارق جزء من الثانية
فقط)، فصندوقاه في الإطارين سيتداخلان بشدة. نقيس هذا التداخل بـ
<strong>IoU</strong> نفسها التي استخدمناها الأسبوع الماضي لتقييم الكشف -
لكن هنا نستخدمها للربط بدل التقييم.</p>

</div>


In [ ]:
def iou(box_a, box_b):
    """كل صندوق بصيغة (x1, y1, x2, y2)."""
    inter_x1 = max(box_a[0], box_b[0])
    inter_y1 = max(box_a[1], box_b[1])
    inter_x2 = min(box_a[2], box_b[2])
    inter_y2 = min(box_a[3], box_b[3])

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


track_box = (100, 100, 300, 250)       # صندوق مسار من الإطار السابق
detection_box = (112, 108, 308, 255)   # صندوق كشف في الإطار الحالي

print("IoU بين المسار والكشف:", round(iou(track_box, detection_box), 3))


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.2 مصفوفة التكلفة</h2>

<p>في إطار حقيقي لا يوجد مسار واحد وكشف واحد، بل عدة مسارات وعدة اكتشافات
في آنٍ واحد. نبني <strong>مصفوفة تكلفة</strong>: صف لكل مسار، وعمود لكل
كشف، وكل خانة هي <code>1 - IoU</code> (تكلفة منخفضة = تشابه عالٍ). المطلوب
بعدها: أي مسار يُربَط بأي كشف بحيث يكون <strong>مجموع التكلفة الكلي أصغر
ما يمكن</strong>؟</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/iou_cost_matrix.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/iou_cost_matrix.jpg';"
alt="Figure 1 - Building a cost matrix from IoU between tracks and detections"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.3 خوارزمية Hungarian، محلولة يدوياً</h2>

<p>هذه مسألة تخصيص كلاسيكية، ولها حل مضبوط في زمن معقول:
<strong>خوارزمية Hungarian</strong>. لن نشتق الخوارزمية، لكن سنراها تحل
مثالاً حقيقياً بثلاثة مسارات وثلاثة اكتشافات.</p>

</div>


In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# صفوف = مسارات T1..T3 من الإطار السابق، أعمدة = اكتشافات D1..D3 في الإطار الحالي
iou_matrix = np.array([
    [0.82, 0.10, 0.00],   # T1
    [0.15, 0.75, 0.05],   # T2
    [0.00, 0.20, 0.68],   # T3
])

cost_matrix = 1.0 - iou_matrix

track_indices, detection_indices = linear_sum_assignment(cost_matrix)

print("مصفوفة IoU:")
print(iou_matrix)
print()
for t, d in zip(track_indices, detection_indices):
    print(f"T{t + 1} <-> D{d + 1}   (IoU = {iou_matrix[t, d]:.2f})")

total_iou = iou_matrix[track_indices, detection_indices].sum()
print(f"\nمجموع IoU للحل الأمثل: {total_iou:.2f}")


<div style="text-align:center; margin:24px 0;">
<img
src="media/hungarian_assignment.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/hungarian_assignment.jpg';"
alt="Figure 2 - The Hungarian algorithm solving a 3x3 track-to-detection assignment"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>3.4 لماذا تفشل المطابقة الجشعة؟</h2>

<p>وسيلة أبسط تخطر بالبال: بدل حل المسألة كاملة، لماذا لا نأخذ في كل مرة
أعلى قيمة IoU في المصفوفة كلها، ونثبّت ذلك الربط، ثم نكرّر على ما تبقّى؟
هذه <strong>المطابقة الجشعة Greedy Matching</strong>. المثال التالي يبيّن
لماذا قد تعطي نتيجة أسوأ من الحل الأمثل:</p>

</div>


In [ ]:
# مثال مضاد: مسار T1 قريب جداً من كشفين معاً، ومسار T2 قريب من كشف واحد فقط
iou_matrix_2 = np.array([
    [0.90, 0.85],   # T1: قريب جداً من كلا الكشفين
    [0.85, 0.00],   # T2: قريب من D1 فقط
])

# الحل الجشع: خذ أعلى قيمة في كل المصفوفة أولاً
greedy_pairs = []
remaining = iou_matrix_2.copy()
for _ in range(remaining.shape[0]):
    t, d = np.unravel_index(np.argmax(remaining), remaining.shape)
    greedy_pairs.append((t, d, remaining[t, d]))
    remaining[t, :] = -1
    remaining[:, d] = -1

greedy_total = sum(score for _, _, score in greedy_pairs)
print("الحل الجشع:")
for t, d, score in greedy_pairs:
    print(f"  T{t + 1} <-> D{d + 1}   (IoU = {score:.2f})")
print(f"  مجموع IoU (الجشع)  : {greedy_total:.2f}")

# الحل الأمثل عبر Hungarian
t_idx, d_idx = linear_sum_assignment(1.0 - iou_matrix_2)
optimal_total = iou_matrix_2[t_idx, d_idx].sum()
print("\nالحل الأمثل (Hungarian):")
for t, d in zip(t_idx, d_idx):
    print(f"  T{t + 1} <-> D{d + 1}   (IoU = {iou_matrix_2[t, d]:.2f})")
print(f"  مجموع IoU (الأمثل) : {optimal_total:.2f}")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<div class="note">
<p>المطابقة الجشعة تأخذ <code>T1-D1</code> فوراً لأنها أعلى قيمة في
المصفوفة كلها (0.90)، فتضطر بعدها إلى ربط <code>T2</code> بـ
<code>D2</code> رغم أن IoU بينهما صفر - أي لا علاقة بينهما إطلاقاً.
Hungarian ترى الصورة كاملة وتختار <code>T1-D2</code> و <code>T2-D1</code>
معاً، فيكون المجموع الكلي أعلى رغم أن كل قيمة على حدة أقل من 0.90.</p>
<p><strong>القاعدة:</strong> القرار الأفضل محلياً في كل خطوة على حدة ليس
بالضرورة الأفضل إجمالاً. هذا بالضبط سبب استخدام SORT وأغلب المتتبعات
الحديثة لخوارزمية Hungarian لا للمطابقة الجشعة.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>4. التنبؤ بالحركة: مرشّح كالمان Kalman Filter</h1>

<h2>4.1 الفكرة بحدس بسيط</h2>

<p>IoU وحدها تفترض أن الجسم لا يتحرك كثيراً بين إطارين. هذا صحيح غالباً،
لكنه يفشل تماماً في حالتين: حين يتحرك الجسم بسرعة، وحين <strong>يختفي
لعدة إطارات</strong> بسبب الحجب. في الحالة الثانية لا يوجد صندوق كشف
أصلاً لنقارنه بأي شيء.</p>

<p>الحل: بدل الاكتفاء بآخر موضع معروف، نبني نموذجاً بسيطاً لحركة الجسم
يجيب عن السؤال: <strong>"لو استمرّ هذا الجسم بنفس سرعته، أين سيكون في
الإطار القادم؟"</strong> هذا هو <strong>مرشّح كالمان</strong>. سنأخذه
بمستوى الحدس فقط، دون اشتقاق رياضي كامل.</p>

<h2>4.2 متجه الحالة</h2>

<p>يمثّل مرشّح كالمان في SORT كل مسار بسبعة أرقام:</p>

<p style="text-align:center; direction:ltr;">
<code>[x, y, s, r, vx, vy, vs]</code>
</p>

<ul>
<li><code>x, y</code>: مركز الصندوق.</li>
<li><code>s</code>: مساحة الصندوق (Scale)، و <code>r</code>: نسبة العرض
إلى الارتفاع (تُفترض شبه ثابتة لكل جسم).</li>
<li><code>vx, vy, vs</code>: سرعة تغيّر كل من <code>x</code> و
<code>y</code> و <code>s</code> بين إطار وآخر.</li>
</ul>

<p>أول أربعة أرقام تصف <strong>أين الجسم الآن</strong>، والثلاثة الأخيرة
تصف <strong>كيف يتحرك</strong> - وهذا الجزء الثاني هو ما يمكّننا من
التنبؤ.</p>

<h2>4.3 دورة التنبؤ ثم التحديث</h2>

<p>في كل إطار جديد، يمرّ كل مسار بخطوتين:</p>

<ol>
<li><strong>Predict (تنبأ)</strong>: بافتراض استمرار السرعة الحالية،
احسب أين يُفترض أن يكون الصندوق في هذا الإطار.
<span style="direction:ltr; display:inline-block;">
<code>x_predicted = x + vx</code></span></li>
<li><strong>Update (حدّث)</strong>: إذا وُجد كشف قريب من الموضع المتنبَّأ
به (بحساب IoU كما تعلّمنا)، اربطهما، واستخدم الكشف الحقيقي لتصحيح تقدير
المرشّح وتحديث السرعة.</li>
</ol>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/kalman_predict_update.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/kalman_predict_update.jpg';"
alt="Figure 3 - The predict-update cycle of a Kalman filter across frames"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>4.4 لماذا ينقذنا أثناء الحجب</h2>

<p>هنا تكمن القوة الحقيقية للفكرة: إن اختفى الجسم فجأة خلف حاجز (حافلة
مثلاً) فلن يصلنا أي كشف له لعدة إطارات. لكن مرشّح كالمان لا يتوقف - يستمر
في خطوة <strong>Predict</strong> فقط (بلا Update، لعدم وجود كشف يصحّحه)،
فيواصل تحريك الصندوق المتوقَّع بنفس السرعة الأخيرة المعروفة.</p>

<p>حين يخرج الجسم من الحجب ويظهر كشف جديد، يكون الصندوق المتوقَّع من
المرشّح قريباً بما يكفي من الكشف الحقيقي لتتطابق معه IoU، فيُربَطان معاً
بنفس الهوية القديمة - <strong>رغم انقطاع الكشف تماماً في الوسط.</strong>
بدون هذا التنبؤ، كان المتتبع سيعتبر ظهور الجسم مجدداً "جسماً جديداً
كلياً" ويعطيه هوية مختلفة.</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/kalman_occlusion.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/kalman_occlusion.jpg';"
alt="Figure 4 - How Kalman prediction bridges a full occlusion without losing the track ID"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>5. SORT</h1>

<p><strong>SORT</strong> (Simple Online and Realtime Tracking) هو أبسط
متتبع عملي، ويجمع كل ما تعلّمناه في القسمين السابقين: كشف، ثم مرشّح كالمان
للتنبؤ، ثم Hungarian للربط. يستحق دراسته لأن كل متتبع أحدث - بما فيها
ByteTrack التي سنستخدمها فعلياً - هو تطوير على هيكله الأساسي.</p>

<h2>5.1 خط الأنابيب كاملاً</h2>

<p>في كل إطار جديد، تمرّ دورة SORT بخطوات ثابتة:</p>

<ol>
<li><strong>Detect</strong>: شغّل الكاشف على الإطار الحالي، فتحصل على قائمة
صناديق.</li>
<li><strong>Predict</strong>: لكل مسار موجود من الإطار السابق، اطلب من
مرشّح كالمان الخاص به موضعه المتوقَّع في هذا الإطار.</li>
<li><strong>Associate</strong>: ابنِ مصفوفة تكلفة IoU بين المواضع
المتوقَّعة والاكتشافات الجديدة، وحلّها بخوارزمية Hungarian.</li>
<li><strong>Update</strong>: كل مسار ارتُبط بكشف يُحدَّث موضعه وسرعته
الحقيقية من ذلك الكشف.</li>
<li><strong>Create</strong>: كل كشف لم يُربَط بأي مسار يبدأ مساراً جديداً
مرشّحاً (Tentative).</li>
<li><strong>Delete</strong>: كل مسار لم يُربَط بأي كشف لفترة طويلة جداً
يُحذف نهائياً.</li>
</ol>

<p>لاحظ أن هذه الدورة هي حرفياً تجميع للأدوات الثلاث من الأقسام 3 و 4:
IoU، Hungarian، ومرشّح كالمان. SORT لا يضيف فكرة جديدة، بل يرتّب الأفكار
الموجودة في خط إنتاج واحد متكرر.</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/sort_pipeline.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/sort_pipeline.jpg';"
alt="Figure 5 - The full SORT pipeline: detect, predict, associate, update, create, delete"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>5.2 دورة حياة المسار: min_hits و max_age</h2>

<p>لا يصبح كل مسار جديد "رسمياً" فوراً، ولا يُحذف كل مسار فقد كشفاً واحداً.
معاملان يضبطان هذا التوازن:</p>

<ul>
<li><strong><code>min_hits</code></strong>: عدد الإطارات المتتالية التي
يجب أن يُطابَق فيها المسار الجديد قبل أن يُعتبر "مؤكَّداً" ويُعرَض
بهويته. يمنع هذا كشفاً عابراً وهمياً (Flicker) من الحصول على هوية دائمة.</li>
<li><strong><code>max_age</code></strong>: عدد الإطارات التي يُسمح للمسار
بالبقاء بلا أي كشف مطابق قبل حذفه نهائياً. رفعه يمنح المسار فرصة أطول
للنجاة من حجب طويل، لكنه أيضاً يُبقي مسارات وهمية حيّة لفترة أطول.</li>
</ul>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/track_lifecycle_state_machine.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/track_lifecycle_state_machine.jpg';"
alt="Figure 6 - Track lifecycle state machine: Tentative to Confirmed to Deleted"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>5.3 نقطة ضعف SORT: تبديل الهوية</h2>

<p>SORT يعتمد <strong>على الحركة فقط</strong>: IoU وموضع متوقَّع من مرشّح
كالمان. لا معلومة عن شكل الجسم أو لونه إطلاقاً. هذا يكفي في أغلب الحالات،
لكنه ينهار في حالة شائعة: <strong>جسمان متشابهان يتقاطعان أو يتقاربان
بشدة</strong>. حين تتداخل الصناديق المتوقَّعة لجسمين قريبين، قد تربط
Hungarian كل مسار بالكشف الخطأ - فتتبادل السيارتان هويتيهما دون أن يخطئ
أي جزء من الخوارزمية على حدة.</p>

<p>هذا يُسمّى <strong>ID Switch</strong>، وهو المقياس الذي سنراه لاحقاً في
القسم 14، وهو بالضبط ما يدفعنا لدراسة DeepSORT وByteTrack.</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/id_switch_example.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/id_switch_example.jpg';"
alt="Figure 7 - An ID switch: two similar objects crossing paths swap identities"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>6. DeepSORT: إضافة المظهر</h1>

<h2>6.1 تمثيلات Re-ID</h2>

<p>فكرة DeepSORT مباشرة: بما أن الحركة وحدها لا تكفي لتمييز جسمين
متشابهين، أضِف معلومة <strong>شكل الجسم نفسه</strong>. لكل صندوق كشف، مرّره
عبر شبكة عصبية صغيرة مدرَّبة خصيصاً لهذه المهمة (تسمى شبكة Re-ID)، فتنتج
<strong>متجه سمات (Embedding)</strong> - مجموعة أرقام تلخّص لون الجسم
وملمسه وشكله العام. سيارتان بيضاوان متجاورتان قد تتشابه صندوقاهما، لكن
متجهيهما سيختلفان قليلاً إن اختلف الطراز أو زاوية الإضاءة.</p>

<h2>6.2 التكلفة المُركَّبة</h2>

<p>بدل الاعتماد على IoU وحدها في مصفوفة التكلفة، يجمع DeepSORT بين
مصدرين:</p>

<ul>
<li><strong>تكلفة الحركة</strong>: المسافة بين الموضع المتوقَّع من مرشّح
كالمان والكشف الجديد (تماماً كما في SORT).</li>
<li><strong>تكلفة المظهر</strong>: المسافة (عادة Cosine Distance) بين
متجه سمات الكشف الجديد ومتوسط متجهات المسار من الإطارات الأخيرة.</li>
</ul>

<p>حين تتقاطع حركتا جسمين فتصبح تكلفة الحركة متعادلة تقريباً بينهما، تكسر
تكلفة المظهر التعادل: الجسم الأبيض يبقى مرتبطاً بمتجه المظهر الأبيض حتى
لو تشابهت موضعيهما لحظياً.</p>

<h2>6.3 الثمن: السرعة</h2>

<p>هذه القوة الإضافية ليست مجانية. تشغيل شبكة Re-ID على <strong>كل صندوق
مكتشَف في كل إطار</strong> يضيف عبء استدلال حقيقياً فوق عبء الكاشف نفسه.
على معالج ضعيف أو مع عدد كبير من الأجسام في المشهد، قد يتحوّل هذا العبء
الإضافي إلى عنق الزجاجة الفعلي للنظام بأكمله - وهو ثمن قد لا يستحقه مشهد
بسيط لا تتقاطع فيه الأجسام كثيراً.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>7. ByteTrack: الفكرة البسيطة الذكية</h1>

<h2>7.1 المشكلة: نرمي الاكتشافات الضعيفة</h2>

<p>تذكّر الأسبوع الماضي: نضبط عتبة <code>conf</code> لنتجاهل الاكتشافات
الضعيفة الثقة، لأن أغلبها إنذارات كاذبة. لكن هذا القرار له ثمن خفي في
التتبع: حين يبدأ جسم حقيقي بالاختفاء تدريجياً خلف حاجز، لا تسقط ثقة
الكاشف به إلى صفر فجأة - بل تنخفض تدريجياً (0.9 ثم 0.6 ثم 0.3...) قبل أن
تختفي تماماً. تلك الاكتشافات الضعيفة في المنتصف <strong>حقيقية غالباً</strong>،
لكن SORT وDeepSORT يرميانها فوراً لأنها تحت العتبة - أي يفقدان الجسم في
اللحظة التي يحتاجان فيها أكثر ما يحتاجان إلى أي دليل عليه.</p>

<h2>7.2 الربط على مرحلتين</h2>

<p>فكرة ByteTrack بسيطة وذكية: <strong>لا ترمِ شيئاً، واستخدمه في مرحلة
ثانية.</strong></p>

<ol>
<li><strong>المرحلة الأولى</strong>: اربط المسارات بالاكتشافات
<strong>عالية الثقة</strong> فقط، بنفس طريقة SORT تماماً.</li>
<li><strong>المرحلة الثانية</strong>: خذ المسارات التي <strong>لم</strong>
تُربَط في المرحلة الأولى (مرشَّحة لأن تكون محجوبة)، وحاول ربطها بالاكتشافات
<strong>منخفضة الثقة</strong> التي رميناها في الطرق التقليدية - باستخدام
IoU فقط، دون شرط الثقة إطلاقاً.</li>
</ol>

<p>النتيجة: مسار كان سيُحذف في المرحلة الأولى (لعدم وجود كشف عالي الثقة
يطابقه) يحصل على فرصة أخيرة في المرحلة الثانية، بدليل ضعيف لكنه أفضل من
لا شيء.</p>

</div>


<div style="text-align:center; margin:24px 0;">
<img
src="media/bytetrack_two_stage.jpg"
onerror="this.onerror=null;this.src='https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/Week8/media/bytetrack_two_stage.jpg';"
alt="Figure 8 - ByteTrack's two-stage association: high-confidence first, then low-confidence recovery"
width="900">
</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h2>7.3 لماذا ينجو من الحجب</h2>

<p>هذا يحل بالضبط المشكلة التي واجهناها في القسم 4: الحجب الجزئي (لا
الكلي) ينتج اكتشافات ضعيفة الثقة لا اكتشافات معدومة. ByteTrack يستغل هذه
المنطقة الرمادية بدل تجاهلها، فيمدّد عمر المسار عبر لحظات الحجب الجزئي
دون الحاجة إلى شبكة Re-ID مكلفة كما في DeepSORT - وهذا سرّ شعبيته: يحسّن
النجاة من الحجب بسرعة SORT نفسها تقريباً.</p>

<h2>7.4 مقارنة: SORT مقابل DeepSORT مقابل ByteTrack مقابل BoT-SORT</h2>

<table>
<thead>
<tr><th></th><th>SORT</th><th>DeepSORT</th><th>ByteTrack</th><th>BoT-SORT</th></tr>
</thead>
<tbody>
<tr><td>مصدر الربط</td><td>حركة (IoU)</td><td>حركة + مظهر (Re-ID)</td>
    <td>حركة (IoU) على مرحلتين</td><td>حركة + تعويض حركة الكاميرا، ومظهر اختياري</td></tr>
<tr><td>يستخدم اكتشافات ضعيفة الثقة</td><td>لا</td><td>لا</td>
    <td>نعم - هذا جوهر الفكرة</td><td>نعم (موروث من ByteTrack)</td></tr>
<tr><td>مقاومة تبديل الهوية</td><td>ضعيفة</td><td>جيدة</td>
    <td>جيدة</td><td>الأفضل عادة</td></tr>
<tr><td>السرعة</td><td>الأسرع</td><td>أبطأ (عبء Re-ID)</td>
    <td>سريع، قريب من SORT</td><td>سريع، أبطأ قليلاً إن فُعِّل Re-ID</td></tr>
<tr><td>ماذا يضيف فوق سابقه</td><td>-</td><td>المظهر</td>
    <td>استغلال الاكتشافات الضعيفة</td><td>تعويض حركة الكاميرا + مظهر اختياري</td></tr>
</tbody>
</table>

<div class="note">
<p>مكتبة Ultralytics التي سنستخدمها في القسم التالي تشحن ByteTrack
و BoT-SORT جاهزين (<code>bytetrack.yaml</code> و <code>botsort.yaml</code>)،
ولن نحتاج إلى تنزيل أي شيء إضافي لتجربتهما.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>8. عملياً مع Ultralytics</h1>

<p>كل ما درسناه نظرياً - IoU، Hungarian، مرشّح كالمان، ByteTrack - مطبَّق
جاهزاً داخل مكتبة Ultralytics نفسها التي استخدمناها للكشف. لا نحتاج إلى
كتابة أي من هذا بأيدينا لنستخدمه فعلياً.</p>

<h2>8.1 model.track() و persist=True</h2>

<p>الفرق الوحيد بين الكشف والتتبع في الاستخدام هو استبدال
<code>model.predict()</code> بـ <code>model.track()</code>، مع تمرير
<code>persist=True</code> ليتذكّر المتتبع مساراته من إطار للذي يليه بدل
البدء من الصفر في كل مرة.</p>

</div>


In [ ]:
AMBULANCE_CLIP = "videos/ambulance_clip.mp4"

first_result = next(model.track(
    source=AMBULANCE_CLIP,
    conf=0.25,
    device=DEVICE,
    tracker="bytetrack.yaml",
    persist=True,
    stream=True,
    verbose=False,
))

print("عدد الاكتشافات في أول إطار:", len(first_result.boxes))
print("هويات التتبع (track ID) لهذا الإطار:", first_result.boxes.id)

plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(first_result.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("model.track() - each box now carries a track ID")
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>لاحظ أن <code>result.plot()</code> رسم رقم الهوية تلقائياً فوق كل
صندوق - <code>result.boxes.id</code> هو الإضافة الوحيدة الجديدة على كل ما
نعرفه من الأسبوع الماضي.</p>

<h2>8.2 bytetrack.yaml مقابل botsort.yaml</h2>

<p>تشحن Ultralytics ملفَي إعداد جاهزين. القيم الافتراضية المهمة في
كليهما:</p>

<table>
<thead>
<tr><th>المعامل</th><th>القيمة الافتراضية</th><th>ماذا يضبط</th></tr>
</thead>
<tbody>
<tr><td><code>track_high_thresh</code></td><td>0.25</td>
    <td>عتبة المرحلة الأولى (عالية الثقة)</td></tr>
<tr><td><code>track_low_thresh</code></td><td>0.10</td>
    <td>عتبة المرحلة الثانية (منخفضة الثقة)</td></tr>
<tr><td><code>track_buffer</code></td><td>30</td>
    <td>عدد الإطارات التي يبقى فيها المسار المفقود حياً</td></tr>
<tr><td><code>match_thresh</code></td><td>0.8</td>
    <td>عتبة تشابه الربط (IoU/تكلفة)</td></tr>
</tbody>
</table>

<p>الفرق الأساسي: <code>botsort.yaml</code> يضيف تعويض حركة الكاميرا
(<code>gmc_method</code>)، ويمكنه تفعيل مظهر Re-ID - لكن
<code>with_reid: False</code> افتراضياً، وهذا مهم: <strong>تفعيله يحمّل
نموذجاً إضافياً من الإنترنت</strong>، فسيكسر العمل بلا اتصال إن فُعِّل.
سنستخدمه هنا بإعداده الافتراضي فقط.</p>

<p>بما أن كاميرتنا ثابتة (لا حركة كاميرا لنعوّضها) و Re-ID معطَّل
افتراضياً في كليهما، هل يُحدِث اختيار أحدهما فرقاً فعلياً على مقطعنا؟
لنقِس، لا نخمّن:</p>

</div>


In [ ]:
def count_unique_ambulance_ids(tracker_yaml):
    ids = set()
    for result in model.track(
        source=AMBULANCE_CLIP, conf=0.25, device=DEVICE,
        tracker=tracker_yaml, persist=True, stream=True, verbose=False,
    ):
        if result.boxes is None or result.boxes.id is None:
            continue
        for track_id, class_id in zip(result.boxes.id, result.boxes.cls):
            if model.names[int(class_id)] == "ambulance":
                ids.add(int(track_id))
    return len(ids)


for tracker_yaml in ["bytetrack.yaml", "botsort.yaml"]:
    n_ids = count_unique_ambulance_ids(tracker_yaml)
    print(f"{tracker_yaml:16s} -> عدد هويات ambulance المختلفة عبر كامل المقطع: {n_ids}")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<div class="note">
<p>النتيجتان متساويتان هنا فعلياً، وهذا متوقَّع: كاميرا هذا المقطع ثابتة
تماماً، فتعويض حركة الكاميرا في BoT-SORT لا يغيّر شيئاً، و Re-ID معطَّل في
كليهما. الفرق بين الاثنين يظهر فقط حين تتحرك الكاميرا نفسها (مثبّتة على
طائرة مسيّرة مثلاً) أو حين نفعّل Re-ID يدوياً.</p>
</div>

<h2>8.3 من boxes.id إلى جدول بيانات</h2>

<p>لتحليل التتبع لاحقاً (كما سنفعل في القسم 13) نحتاج تجميع كل الاكتشافات
المتتبَّعة عبر المقطع كاملاً في جدول واحد:</p>

</div>


In [ ]:
tracking_rows = []

for frame_index, result in enumerate(model.track(
    source=AMBULANCE_CLIP, conf=0.25, device=DEVICE,
    tracker="bytetrack.yaml", persist=True, stream=True, verbose=False,
)):
    if result.boxes is None or result.boxes.id is None:
        continue
    for box, track_id, class_id, confidence in zip(
        result.boxes.xyxy, result.boxes.id, result.boxes.cls, result.boxes.conf
    ):
        tracking_rows.append({
            "frame": frame_index,
            "track_id": int(track_id),
            "class": model.names[int(class_id)],
            "confidence": round(float(confidence), 2),
        })

tracking_df = pd.DataFrame(tracking_rows)
print("عدد الصفوف:", len(tracking_df))
print()
print("عدد هويات التتبع المختلفة لكل فئة:")
print(tracking_df.groupby("class")["track_id"].nunique())
tracking_df.head()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>النتيجة تكشف مشكلة حقيقية سنعود إليها في القسم 13: يوجد فعلياً
<strong>إسعاف واحد فقط</strong> في هذا المقطع، لكن الجدول أعلاه يُظهر
عشرات الهويات المختلفة له - أي أن التتبع يفقد الهوية ويعيد اختراعها مراراً.
هذا ليس خطأً في الكود، بل انعكاس صادق لضعف كشف بعض الفئات كما رأيناه في
الأسبوع الماضي.</p>

<h2>8.4 لماذا stream=True؟</h2>

<p>مرّرنا <code>stream=True</code> في كل الأمثلة أعلاه. بدونها تحاول
Ultralytics معالجة الفيديو كاملاً أولاً ثم إرجاع كل النتائج دفعة واحدة في
الذاكرة - وهذا قد يستهلك ذاكرة هائلة مع فيديو طويل. مع
<code>stream=True</code> تُعالَج الإطارات واحداً تلو الآخر كمولِّد
(Generator)، فتبقى الذاكرة المستخدمة صغيرة وثابتة بغض النظر عن طول
الفيديو.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>9. نبني متتبع IoU مصغّراً من الصفر</h1>

<p>ByteTrack يعمل جيداً، لكنه صندوق أسود. لنبنِ أبسط متتبع ممكن بأيدينا -
باستخدام IoU و Hungarian فقط من القسم 3، <strong>بلا مرشّح كالمان
إطلاقاً</strong> - لنرى بالضبط أين تنهار الفكرة البسيطة، وبالتالي لماذا
احتجنا كل ما تعلّمناه بعدها.</p>

</div>


In [ ]:
class SimpleIoUTracker:
    """متتبع مصغّر: مطابقة جشعة عبر Hungarian على IoU فقط، بلا تنبؤ حركة."""

    def __init__(self, max_age=5, iou_threshold=0.3):
        self.max_age = max_age
        self.iou_threshold = iou_threshold
        self.tracks = {}   # track_id -> {"box": ..., "age": ..., "class": ...}
        self.next_id = 1

    def update(self, boxes, classes):
        if not self.tracks:
            for box, class_name in zip(boxes, classes):
                self.tracks[self.next_id] = {"box": box, "age": 0, "class": class_name}
                self.next_id += 1
            return

        track_ids = list(self.tracks.keys())
        cost_matrix = np.ones((len(track_ids), len(boxes)))
        for i, track_id in enumerate(track_ids):
            for j, box in enumerate(boxes):
                cost_matrix[i, j] = 1 - iou(self.tracks[track_id]["box"], box)

        matched_tracks, matched_dets = ([], [])
        if len(track_ids) and len(boxes):
            matched_tracks, matched_dets = linear_sum_assignment(cost_matrix)

        assigned_dets, assigned_tracks = set(), set()
        for t, d in zip(matched_tracks, matched_dets):
            if cost_matrix[t, d] <= 1 - self.iou_threshold:
                track_id = track_ids[t]
                self.tracks[track_id]["box"] = boxes[d]
                self.tracks[track_id]["age"] = 0
                assigned_tracks.add(track_id)
                assigned_dets.add(d)

        for track_id in track_ids:
            if track_id not in assigned_tracks:
                self.tracks[track_id]["age"] += 1
        self.tracks = {tid: t for tid, t in self.tracks.items() if t["age"] <= self.max_age}

        for j, (box, class_name) in enumerate(zip(boxes, classes)):
            if j not in assigned_dets:
                self.tracks[self.next_id] = {"box": box, "age": 0, "class": class_name}
                self.next_id += 1


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<div class="note">
<p><strong>نعيد تحميل النموذج هنا في متغيّر جديد <code>(fresh_model)</code>
بدل استخدام <code>model</code> نفسه.</strong> السبب فخّ اكتشفناه أثناء إعداد
هذا الدرس: استدعاء <code>model.track(..., persist=True)</code> كما فعلنا
في القسم 8 <strong>يُغيّر داخلياً حالة الكاشف المخزَّنة في الكائن</strong>،
فتختلف نتائج <code>model.predict()</code> اللاحقة على نفس الفيديو بصمت -
عدد اكتشافات الإسعاف الخام تحوّل فعلياً من 520 إلى 438 في اختبارنا لهذا
بالضبط. تحميل نسخة جديدة من النموذج قبل أي مقارنة حساسة للأرقام يتجنّب هذا
الفخّ تماماً.</p>
</div>

<p>نشغّل متتبعنا على مقطع الإسعاف كاملاً، ونعدّ <strong>كل</strong> هوية
مختلفة أُنشئت لفئة <code>ambulance</code> عبر حياة التشغيل كلها - لا فقط
الهويات الحيّة في النهاية:</p>

</div>


In [ ]:
fresh_model = YOLO(MODEL_PATH)  # نسخة جديدة، بلا أي تأثير من استدعاءات track() السابقة

simple_tracker = SimpleIoUTracker(max_age=5, iou_threshold=0.3)
ever_created = {}

for result in fresh_model.predict(source=AMBULANCE_CLIP, conf=0.25, device=DEVICE, stream=True, verbose=False):
    boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.zeros((0, 4))
    classes = [fresh_model.names[int(c)] for c in result.boxes.cls] if result.boxes is not None else []

    existing_ids = set(simple_tracker.tracks.keys())
    simple_tracker.update(boxes, classes)
    for new_id in set(simple_tracker.tracks.keys()) - existing_ids:
        ever_created[new_id] = simple_tracker.tracks[new_id]["class"]

n_ambulance_ids = sum(1 for class_name in ever_created.values() if class_name == "ambulance")
print("عدد هويات (ambulance) المختلفة التي أنشأها SimpleIoUTracker:", n_ambulance_ids)
print("(للمقارنة: ByteTrack في القسم 8.2 أعطى 22 هوية على المقطع نفسه)")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<div class="note">
<p><strong>متتبعنا المصغّر أسوأ من ByteTrack، وهذا متوقَّع تماماً.</strong>
بلا مرشّح كالمان، أي إطار يُفقَد فيه الكشف (ولو لحظياً) يعني أن أقرب صندوق
تالٍ قد لا يتقاطع كفاية مع آخر موضع معروف، فتُفتح هوية جديدة كلياً بدل
استكمال القديمة. هذا بالضبط ما شرحناه بالحدس في القسم 4.4 - والآن رأيناه
رقماً حقيقياً لا افتراضاً نظرياً.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>10. رسم المسارات Trails وتلوين حسب الـ ID</h1>

<p>بما أن كل جسم له هوية ثابتة الآن، يمكننا رسم <strong>أثر حركته</strong>
عبر الإطارات الأخيرة - خط يتتبّع مركز صندوقه، بلون ثابت طوال حياة تلك
الهوية. سنستخدم ByteTrack (لا متتبعنا المصغّر) لأن الهدف هنا عرض المسارات
لا اختبار حدود المطابقة.</p>

<p>بدل عرض فيديو كاملاً (وهو ثقيل داخل دفتر يُقرأ دون تشغيله - راجع
HANDOFF.md #5.3)، سنأخذ خمسة إطارات موزَّعة عبر المقطع ونعرضها جنباً إلى
جنب، فنرى استقرار الهوية عبر الزمن دفعة واحدة بدل حركة فيديو.</p>

</div>


In [ ]:
SNAPSHOT_FRAMES = {90, 200, 300, 340, 380}
TRAIL_LENGTH = 40

track_history = {}
palette_bgr = [(198, 40, 40), (21, 101, 192), (46, 125, 50),
               (249, 168, 37), (123, 31, 162), (0, 131, 143)]
snapshots = {}

for frame_index, result in enumerate(model.track(
    source=AMBULANCE_CLIP, conf=0.25, device=DEVICE,
    tracker="bytetrack.yaml", persist=True, stream=True, verbose=False,
)):
    annotated_frame = result.orig_img.copy()

    if result.boxes is not None and result.boxes.id is not None:
        for box, track_id, class_id in zip(result.boxes.xyxy, result.boxes.id, result.boxes.cls):
            track_id = int(track_id)
            x1, y1, x2, y2 = map(int, box)
            center = ((x1 + x2) // 2, (y1 + y2) // 2)
            color = palette_bgr[track_id % len(palette_bgr)]

            track_history.setdefault(track_id, []).append(center)
            track_history[track_id] = track_history[track_id][-TRAIL_LENGTH:]

            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(annotated_frame, f"ID {track_id}", (x1, max(y1 - 6, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)

            trail_points = track_history[track_id]
            for i in range(1, len(trail_points)):
                cv2.line(annotated_frame, trail_points[i - 1], trail_points[i], color, 2)

    if frame_index in SNAPSHOT_FRAMES:
        snapshots[frame_index] = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, len(SNAPSHOT_FRAMES), figsize=(18, 4))
for ax, frame_index in zip(axes, sorted(snapshots)):
    ax.imshow(snapshots[frame_index])
    ax.set_title(f"frame {frame_index}", fontsize=11)
    ax.axis("off")
plt.suptitle("Same ambulance, same ID, across the clip", fontsize=14)
plt.tight_layout()
plt.show()


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>لاحظ خط الأثر خلف كل صندوق: هو تجسيد بصري مباشر لما تعنيه "هوية
ثابتة عبر الزمن" - شيء لم يكن ممكناً إطلاقاً بالكشف وحده.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>11. تشغيل على المقطع الكامل وحفظ الناتج</h1>

<p>الآن نُخرج النتيجة كفيديو كامل نحفظه على القرص، بدل شرائح ثابتة فقط.
نستخدم <code>imageio-ffmpeg</code> بدل الاعتماد على وجود ffmpeg مثبَّتاً
في نظام الطالب - راجع HANDOFF.md #9.6.</p>

<div class="note">
<p><strong>لن نعرض الفيديو داخل الدفتر</strong> عبر
<code>Video(path, embed=True)</code>، لأن هذا يحوّل الفيديو إلى نص
Base64 ضخم يُضاعف حجم ملف الدفتر أضعافاً (وهذا بالضبط ما ضخّم دفتر
الأسبوع السادس). الفيديو يُحفظ في <code>videos/reference_tracking_output.mp4</code>
ويُفتح من هناك مباشرة.</p>
</div>

</div>


In [ ]:
import imageio_ffmpeg

OUTPUT_VIDEO = "videos/reference_tracking_output.mp4"

capture = cv2.VideoCapture(AMBULANCE_CLIP)
video_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = capture.get(cv2.CAP_PROP_FPS)
capture.release()

writer = imageio_ffmpeg.write_frames(
    OUTPUT_VIDEO, (video_width, video_height), fps=video_fps,
    codec="libx264", macro_block_size=1,
    output_params=["-crf", "28", "-pix_fmt", "yuv420p"],
)
writer.send(None)

track_history = {}

for result in model.track(
    source=AMBULANCE_CLIP, conf=0.25, device=DEVICE,
    tracker="bytetrack.yaml", persist=True, stream=True, verbose=False,
):
    annotated_frame = result.orig_img.copy()
    if result.boxes is not None and result.boxes.id is not None:
        for box, track_id, class_id in zip(result.boxes.xyxy, result.boxes.id, result.boxes.cls):
            track_id = int(track_id)
            x1, y1, x2, y2 = map(int, box)
            center = ((x1 + x2) // 2, (y1 + y2) // 2)
            color = palette_bgr[track_id % len(palette_bgr)]

            track_history.setdefault(track_id, []).append(center)
            track_history[track_id] = track_history[track_id][-TRAIL_LENGTH:]

            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
            label = f"{model.names[int(class_id)]} {track_id}"
            cv2.putText(annotated_frame, label, (x1, max(y1 - 6, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)

            trail_points = track_history[track_id]
            for i in range(1, len(trail_points)):
                cv2.line(annotated_frame, trail_points[i - 1], trail_points[i], color, 2)

    writer.send(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB).tobytes())

writer.close()

output_size_mb = Path(OUTPUT_VIDEO).stat().st_size / 1e6
print(f"حُفظ الفيديو في: {OUTPUT_VIDEO}")
print(f"الحجم: {output_size_mb:.2f} MB")


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>افتح <code>Week8/videos/reference_tracking_output.mp4</code> مباشرة
من مستكشف الملفات لمشاهدته كاملاً. طوله خمس عشرة ثانية، وحجمه أقل من
2 ميغابايت.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>12. لمحة أولى: كيف يفتح التتبع باب العدّ؟</h1>

<p>عد الآن إلى مشكلة العدّ المضاعف التي بدأنا بها الدرس في القسم 1: كنا
نجمع "عدد السيارات المكتشفة" في كل إطار، فحصلنا على رقم سخيف
(956 على مقطع من 15 ثانية) لأن الكشف لا يعرف أن السيارة نفسها تظهر مراراً.</p>

<p>الآن وقد أصبح لكل جسم <strong>هوية ثابتة</strong>، يتغيّر السؤال كلياً:
بدل "كم اكتشافاً رأيت؟" نستطيع أن نسأل <strong>"كم هوية مختلفة رأيت؟"</strong>
- وهذا أقرب بما لا يُقاس إلى العدّ الحقيقي. لو رسمنا خطاً افتراضياً عبر
الطريق، وسجّلنا كل مرة تعبره <em>هوية جديدة لم نرها من قبل</em>، لحصلنا
على عدّاد مركبات معقول.</p>

<div class="note">
<p>هذا هو بالضبط الجسر إلى الأسبوع القادم. بناء عدّاد كامل بخط عبور
واتجاه حركة وواجهة عرض يحتاج تفصيلاً إضافياً - كثافة المرور، واتجاه كل
حارة، ومنطق القرار عند تعدد الحارات - وهذا كله موضوع الأسبوع التاسع.
ما يهمّنا اليوم هو الفكرة الجوهرية فقط: <strong>لا عدّ صحيح بلا هوية
ثابتة</strong>، وقد بنينا تلك الهوية للتو.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>13. متى يفشل المتتبع؟</h1>

<h2>13.1 ثلاثة أنواع فشل مختلفة</h2>

<table>
<thead>
<tr><th>النوع</th><th>ماذا يحدث</th><th>رأيناه فعلياً في</th></tr>
</thead>
<tbody>
<tr>
  <td><strong>تبديل الهوية ID Switch</strong></td>
  <td>جسمان يتبادلان هويتيهما عند التقاطع أو الاقتراب الشديد</td>
  <td>القسم 5.3 (SORT، نظرياً)</td>
</tr>
<tr>
  <td><strong>التجزّؤ Fragmentation</strong></td>
  <td>جسم واحد فعلياً يُعطى عدة هويات متتالية بدل هوية واحدة مستمرة</td>
  <td>القسم 9: 29 هوية <code>ambulance</code> لإسعاف واحد فعلي</td>
</tr>
<tr>
  <td><strong>مسار وهمي Ghost Track</strong></td>
  <td>يبقى مسار حياً رغم اختفاء الجسم الحقيقي، بسبب اكتشاف خاطئ متكرر</td>
  <td>خطر متزايد كلما رفعنا <code>track_buffer</code></td>
</tr>
</tbody>
</table>

<p>لاحظ أن ما رأيناه فعلياً في هذا الدرس هو <strong>التجزّؤ</strong> بالدرجة
الأولى، لا تبديل الهوية. السبب: مصدر ضعفنا هنا ليس تقاطع أجسام متشابهة، بل
كشف متذبذب لفئة <code>ambulance</code> نفسها (كما وثّقنا الأسبوع الماضي).
هذا درس مهم: <strong>نوع فشل التتبع الذي تراه يعتمد على نوع ضعف الكاشف الذي
تغذّيه به</strong>.</p>

<h2>13.2 نرى التجزّؤ بالأرقام</h2>

<p>نعود إلى <code>tracking_df</code> الذي بنيناه في القسم 8.3، ونفحص كل
هوية <code>ambulance</code> على حدة: متى ظهرت، ومتى اختفت، وكم إطاراً
عاشت؟</p>

</div>


In [ ]:
ambulance_tracks = tracking_df[tracking_df["class"] == "ambulance"]

lifespans = ambulance_tracks.groupby("track_id")["frame"].agg(["min", "max", "count"])
lifespans.columns = ["أول إطار", "آخر إطار", "عدد الإطارات"]
lifespans = lifespans.sort_values("أول إطار")

print(f"عدد الهويات المختلفة لفئة ambulance: {len(lifespans)}")
print(f"متوسط عمر الهوية الواحدة: {lifespans['عدد الإطارات'].mean():.1f} إطار فقط")
print(f"أطول هوية عاشت: {lifespans['عدد الإطارات'].max()} إطاراً")
lifespans.head(10)


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<p>معظم الهويات تعيش عدداً قليلاً جداً من الإطارات قبل أن تُفقَد
وتُستبدَل بهوية جديدة. هذا هو التجزّؤ حرفياً: نفس الإسعاف الفعلي يُعاد
تعريفه عشرات المرات بدل أن يُعرَّف مرة واحدة ويستمر.</p>

<h2>13.3 جدول الضبط: المعاملات التي تتحكّم بهذا التوازن</h2>

<table>
<thead>
<tr><th>المعامل</th><th>الافتراضي</th><th>رفعه يعني</th><th>خفضه يعني</th></tr>
</thead>
<tbody>
<tr>
  <td><code>track_high_thresh</code></td><td>0.25</td>
  <td>مطابقة أكثر تحفّظاً في المرحلة الأولى</td>
  <td>قبول اكتشافات أضعف كمرحلة أولى، خطر تبديل هوية أعلى</td>
</tr>
<tr>
  <td><code>track_low_thresh</code></td><td>0.10</td>
  <td>مرحلة ثانية أكثر تشدّداً، تجزّؤ أكثر</td>
  <td>إنقاذ أكثر من الحجب، لكن خطر ربط ضجيج بمسار حقيقي</td>
</tr>
<tr>
  <td><code>match_thresh</code></td><td>0.80</td>
  <td>يتطلّب تداخلاً أعلى ليقبل الربط، تجزّؤ أكثر عند الحركة السريعة</td>
  <td>يقبل ربطاً أضعف، خطر تبديل هوية أعلى</td>
</tr>
<tr>
  <td><code>track_buffer</code></td><td>30</td>
  <td>يبقي المسار المفقود حياً فترة أطول، ينجو من حجب أطول</td>
  <td>يحذف المسار المفقود بسرعة، تجزّؤ أكثر عند أي انقطاع قصير</td>
</tr>
</tbody>
</table>

<div class="note">
<p><strong>لا يوجد إعداد "صحيح" مطلق.</strong> كل معامل هنا مقايضة: رفعه
يحلّ مشكلة ويخلق أخرى. الإعداد الأنسب يعتمد على مشهدك: طريق مزدحم بأجسام
متشابهة يحتاج <code>match_thresh</code> أعلى لتفادي تبديل الهوية، بينما
مشهد فيه حجب طويل متكرر يحتاج <code>track_buffer</code> أعلى - حتى لو
كلَّفَ ذلك بعض مسارات الأشباح.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>14. كيف نقيس جودة التتبع؟</h1>

<p>الأسبوع الماضي قِسنا جودة الكشف بـ Precision و Recall و mAP. التتبع
يحتاج مقاييس مختلفة، لأنه يضيف بعداً لم يكن موجوداً من قبل: <strong>هل
الهوية نفسها استمرّت بشكل صحيح عبر الزمن؟</strong> هذا القسم مفاهيمي بحت -
فهمُ الرمز أهم من حسابه بأنفسنا اليوم.</p>

<table>
<thead>
<tr><th>المقياس</th><th>يقيس</th><th>حساسيته لتبديل الهوية</th></tr>
</thead>
<tbody>
<tr>
  <td><strong>MOTA</strong><br/>(Multi-Object Tracking Accuracy)</td>
  <td>يجمع الاكتشافات المفقودة (FN) والخاطئة (FP) وتبديلات الهوية في رقم
  واحد</td>
  <td>منخفضة نسبياً - تبديل هوية واحد يُحتسب بوزن صغير مقارنة بآلاف
  الاكتشافات</td>
</tr>
<tr>
  <td><strong>IDF1</strong></td>
  <td>مدى تطابق الهويات المتوقَّعة مع الهويات الحقيقية عبر الزمن كله، لا
  إطاراً بإطار</td>
  <td>عالية - مصمَّم خصيصاً ليعاقب فقدان الاستمرارية</td>
</tr>
<tr>
  <td><strong>HOTA</strong><br/>(Higher Order Tracking Accuracy)</td>
  <td>يفصل بوضوح بين دقة الكشف ودقة الربط، ثم يجمعهما بتوازن مدروس</td>
  <td>عالية، والأحدث والأكثر توازناً بين المقياسين السابقين</td>
</tr>
</tbody>
</table>

<div class="note">
<p><strong>لماذا لم نحسب أياً من هذه الأرقام على مقاطعنا؟</strong> لأن
حسابها يحتاج <strong>توسيماً حقيقياً لكل إطار</strong>: صندوق كل جسم مع
هويته الصحيحة يدوياً، إطاراً إطاراً، طوال المقطع - وهذا عمل توسيم ضخم يفوق
نطاق هذا الدرس. ما فعلناه بدلاً منه في القسم 13 أبسط لكنه صادق بنفس القدر:
عددنا الهويات المختلفة التي أنشأها المتتبع لجسم نعرف يقيناً أنه واحد فعلياً
- وهذا مؤشر تجزّؤ خام لا يحتاج توسيماً على الإطلاق.</p>
</div>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>15. تمرين صفي</h1>

<h3>المهمة 1: أثر track_buffer</h3>
<p>عدّل قيمة <code>track_buffer</code> في إعداد ByteTrack: جرّبها 5 ثم
100 (بدل الافتراضي 30)، وأعد حساب عدد هويات <code>ambulance</code>
المختلفة كما في القسم 8.2. هل النتيجة كما توقّعت من جدول القسم 13.3؟</p>

</div>


In [ ]:
# TODO: انسخ إعداد bytetrack الافتراضي وعدّل track_buffer فيه، ثم أعد القياس
custom_tracker_yaml = """
tracker_type: bytetrack
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.25
track_buffer: 30
match_thresh: 0.8
fuse_score: True
"""
# 1. غيّر track_buffer أعلاه إلى 5، احفظ الملف، أعد تشغيل count_unique_ambulance_ids
# 2. كرّر بقيمة 100
# 3. سجّل النتيجتين وقارنهما بنتيجة القسم 8.2 (22 مع القيمة الافتراضية 30)

Path("custom_bytetrack.yaml").write_text(custom_tracker_yaml)
print("عدّل الملف أعلاه ثم شغّل:")
print('count_unique_ambulance_ids("custom_bytetrack.yaml")')


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h3>المهمة 2: ByteTrack مقابل BoT-SORT بصرياً</h3>
<p>في القسم 8.2 كانت أعداد الهويات متساوية بين الاثنين. أعد توليد الفيديو
المتتبَّع في القسم 11 مرة بـ <code>tracker="botsort.yaml"</code> بدل
<code>bytetrack.yaml</code>. شاهد الفيديوين جنباً إلى جنب: هل تلاحظ أي فرق
بصري رغم تساوي العدد الإجمالي؟</p>

<h3>المهمة 3: أثر max_age في متتبعنا المصغّر</h3>
<p>في القسم 9، جرّب <code>SimpleIoUTracker(max_age=1)</code> ثم
<code>SimpleIoUTracker(max_age=20)</code> بدل الافتراضي 5. سجّل عدد هويات
<code>ambulance</code> في كل حالة.</p>

</div>


In [ ]:
# TODO: جرّب القيمتين وسجّل النتيجة في كل مرة
for trial_max_age in [1, 20]:
    trial_model = YOLO(MODEL_PATH)
    trial_tracker = SimpleIoUTracker(max_age=trial_max_age, iou_threshold=0.3)
    trial_ever_created = {}

    for result in trial_model.predict(source=AMBULANCE_CLIP, conf=0.25, device=DEVICE, stream=True, verbose=False):
        boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.zeros((0, 4))
        classes = [trial_model.names[int(c)] for c in result.boxes.cls] if result.boxes is not None else []
        existing_ids = set(trial_tracker.tracks.keys())
        trial_tracker.update(boxes, classes)
        for new_id in set(trial_tracker.tracks.keys()) - existing_ids:
            trial_ever_created[new_id] = trial_tracker.tracks[new_id]["class"]

    n_ids = sum(1 for c in trial_ever_created.values() if c == "ambulance")
    print(f"max_age={trial_max_age:3d}  ->  عدد هويات ambulance: {n_ids}")

# فسّر: لماذا يقلّل max_age الكبير عدد الهويات؟ وما ثمنه المحتمل (مسارات وهمية)؟


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<h3>المهمة 4: أوجد لحظة الانهيار بنفسك</h3>
<p>شغّل <code>SimpleIoUTracker</code> على مقطع الإسعاف إطاراً إطاراً، واطبع
رقم الإطار وهوية <code>ambulance</code> الحالية في كل مرة. حدّد
<strong>الإطار بالضبط</strong> الذي تتغيّر فيه الهوية لأول مرة، وافحص تلك
اللحظة بصرياً (هل توقّف الكشف؟ هل تغيّرت زاوية الإسعاف؟). اكتب سطرين
تشرحان السبب الأرجح.</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>16. أسئلة مراجعة سريعة</h1>

<h3>سؤال 1</h3>
<p>لماذا يفشل العدّ الساذج الذي يجمع عدد الاكتشافات في كل إطار على حدة؟</p>

<h3>سؤال 2</h3>
<p>عرّف الفرق بين Detection و Track و ID بكلماتك الخاصة.</p>

<h3>سؤال 3</h3>
<p>لماذا تُفضَّل خوارزمية Hungarian على المطابقة الجشعة في الربط بين
الإطارات؟ اذكر مثالاً يوضّح الفرق.</p>

<h3>سؤال 4</h3>
<p>كيف ينقذنا مرشّح كالمان أثناء الحجب الكامل؟ وما حدود هذا الإنقاذ - متى
يفشل هو الآخر؟</p>

<h3>سؤال 5</h3>
<p>ما الفرق الجوهري بين SORT و DeepSORT و ByteTrack من حيث مصدر معلومة
الربط التي يعتمد عليها كل منها؟</p>

<h3>سؤال 6</h3>
<p>ما الفرق بين تبديل الهوية (ID Switch) والتجزّؤ (Fragmentation)؟ أيّهما
شاهدناه فعلياً في تجربتنا، ولماذا برأيك؟</p>

<h3>سؤال 7</h3>
<p>لماذا لا يمكننا حساب MOTA أو IDF1 أو HOTA على مقاطعنا الخاصة في هذا
الدرس؟ ماذا كان سيتطلّب ذلك؟</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>17. بنك أسئلة Kahoot</h1>

<div class="note">
<p>هذه الأسئلة جاهزة للنسخ إلى Kahoot. الحدود المسموحة في المنصة: نص
السؤال حتى 120 حرفاً، وكل خيار حتى 75 حرفاً، وأربعة خيارات، وإجابة صحيحة
واحدة.</p>
</div>

<h3>سؤال 1</h3>
<p><strong>السؤال:</strong> ما الذي يفتقر إليه الكشف وحده مقارنة بالتتبع؟</p>
<ul>
<li>أ) الدقة</li>
<li>ب) الذاكرة بين الإطارات</li>
<li>ج) السرعة</li>
<li>د) الألوان</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 2</h3>
<p><strong>السؤال:</strong> ما الذي يضيفه التتبع فوق الكشف؟</p>
<ul>
<li>أ) فئة جديدة كلياً</li>
<li>ب) هوية ثابتة تلازم الجسم عبر الزمن</li>
<li>ج) دقة صناديق أعلى</li>
<li>د) نموذجاً أكبر حجماً</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 3</h3>
<p><strong>السؤال:</strong> ما الذي تقيسه IoU بين صندوقين؟</p>
<ul>
<li>أ) الفرق في الألوان بينهما</li>
<li>ب) نسبة التقاطع إلى الاتحاد</li>
<li>ج) سرعة حركة الجسم</li>
<li>د) عدد البكسلات داخل الصندوق</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 4</h3>
<p><strong>السؤال:</strong> لماذا قد تفشل المطابقة الجشعة رغم أن كل خطوة
تبدو منطقية؟</p>
<ul>
<li>أ) لأنها بطيئة جداً في التنفيذ</li>
<li>ب) لأنها لا ترى مصفوفة التكلفة كاملة دفعة واحدة</li>
<li>ج) لأنها تحتاج كرت شاشة قوياً</li>
<li>د) لأنها لا تدعم أكثر من صندوقين</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 5</h3>
<p><strong>السؤال:</strong> ماذا يفعل مرشّح كالمان أثناء حجب كامل للجسم؟</p>
<ul>
<li>أ) يتوقف عن العمل فوراً</li>
<li>ب) يواصل التنبؤ بلا تصحيح من كشف حقيقي</li>
<li>ج) يحذف المسار في الحال</li>
<li>د) يطلب كشفاً جديداً من المستخدم</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 6</h3>
<p><strong>السؤال:</strong> ما الذي يضيفه DeepSORT فوق SORT؟</p>
<ul>
<li>أ) سرعة معالجة أعلى</li>
<li>ب) معلومة المظهر عبر شبكة Re-ID</li>
<li>ج) دقة صناديق أعلى تلقائياً</li>
<li>د) عتبة ثقة أقل للكاشف</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 7</h3>
<p><strong>السؤال:</strong> ما الفكرة الجوهرية في ByteTrack؟</p>
<ul>
<li>أ) تجاهل كل الاكتشافات ضعيفة الثقة</li>
<li>ب) استخدام الاكتشافات الضعيفة بدل رميها فوراً</li>
<li>ج) إضافة كاميرا ثانية للمشهد</li>
<li>د) تدريب نموذج كشف أكبر حجماً</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 8</h3>
<p><strong>السؤال:</strong> أي معامل يضبط مدة بقاء المسار المفقود حياً؟</p>
<ul>
<li>أ) conf</li>
<li>ب) track_buffer</li>
<li>ج) imgsz</li>
<li>د) batch</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 9</h3>
<p><strong>السؤال:</strong> كم هوية ambulance أنشأها متتبعنا المصغّر لإسعاف
واحد فعلياً في تجربتنا؟</p>
<ul>
<li>أ) 1</li>
<li>ب) 5</li>
<li>ج) 29</li>
<li>د) 100</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ج</p>

<h3>سؤال 10</h3>
<p><strong>السؤال:</strong> لماذا تساوى عدد الهويات بين bytetrack و botsort
في مقطعنا؟</p>
<ul>
<li>أ) الكاميرا ثابتة و Re-ID معطَّل في كليهما افتراضياً</li>
<li>ب) الملفّان متطابقان حرفياً بلا أي فرق</li>
<li>ج) خطأ برمجي في خلية المقارنة</li>
<li>د) النموذج لا يفرّق بين المتتبعين إطلاقاً</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> أ</p>

<h3>سؤال 11</h3>
<p><strong>السؤال:</strong> لماذا أعدنا تحميل نموذج جديد قبل قسم 9 بدل
استخدام model نفسه؟</p>
<ul>
<li>أ) للتسريع فقط، بلا أثر على النتائج</li>
<li>ب) لأن track السابق غيّر حالة النموذج بصمت فتختلف النتائج</li>
<li>ج) عادة برمجية شائعة بلا سبب حقيقي</li>
<li>د) لتوفير مساحة تخزين على القرص</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

<h3>سؤال 12</h3>
<p><strong>السؤال:</strong> ما الخطر الأكبر من رفع track_buffer كثيراً؟</p>
<ul>
<li>أ) بطء شديد لا يُحتمل في المعالجة</li>
<li>ب) مسارات وهمية تبقى حيّة طويلاً بلا داعٍ</li>
<li>ج) فقدان ملف الفيديو الأصلي</li>
<li>د) تعطّل النموذج عن العمل تماماً</li>
</ul>
<p><strong>الإجابة الصحيحة:</strong> ب</p>

</div>


<div dir="rtl" class="rtl-cell" style="direction:rtl; text-align:right;">

<hr />
<h1>18. الخلاصة</h1>

<p>ما فعلناه في هذا الدرس:</p>

<ul>
<li>أثبتنا أن العدّ الساذج بجمع اكتشافات كل إطار على حدة يعطي أرقاماً
سخيفة (956 "سيارة" في 15 ثانية)، لأن الكشف عديم الذاكرة بين الإطارات.</li>
<li>بنينا الأدوات الثلاث الأساسية للربط بين الإطارات: IoU، مصفوفة التكلفة،
وخوارزمية Hungarian - ورأينا بالأرقام لماذا تتفوّق على المطابقة الجشعة.</li>
<li>فهمنا بالحدس كيف ينقذنا مرشّح كالمان أثناء الحجب، بالتنبؤ بالحركة بدل
انتظار كشف قد لا يصل أبداً.</li>
<li>قارنّا SORT و DeepSORT و ByteTrack: كل واحد يحلّ نقطة ضعف في سابقه،
بثمن مختلف في كل مرة.</li>
<li>بنينا متتبعاً مصغّراً بأيدينا فأنتج 29 هوية لإسعاف واحد فعلياً - ورأينا
التجزّؤ رقماً حقيقياً لا افتراضاً نظرياً.</li>
<li>استخدمنا <code>model.track()</code> من Ultralytics عملياً، ورسمنا
مسارات وهويات ثابتة، وحفظنا فيديو متتبَّعاً كاملاً على القرص.</li>
<li>اكتشفنا فخّاً حقيقياً في Ultralytics: <code>track(persist=True)</code>
يغيّر حالة النموذج بصمت، فيؤثر على استدعاءات <code>predict()</code>
اللاحقة - ووثّقناه ليتجنّبه غيرنا.</li>
</ul>

<h2>في الأسبوع القادم</h2>

<p>أصبح لدينا الآن ما كان ناقصاً في نهاية الأسبوع السابع: <strong>هوية
ثابتة لكل جسم عبر الزمن</strong>. الأسبوع التاسع يبني على هذا مباشرة:
عدّ حقيقي للمركبات بخط عبور، وتقدير كثافة المرور، ثم منطق القرار الكامل
لنظام "الإسعاف له الأولوية" - من رؤية الإسعاف إلى فتح الإشارة له فعلياً.</p>

<hr />
<h2>مصادر للاستزادة</h2>

<ul>
<li><a href="https://docs.ultralytics.com/modes/track/">Ultralytics — Track mode</a></li>
<li><a href="https://docs.ultralytics.com/reference/trackers/">Ultralytics — Trackers reference</a></li>
<li><a href="https://arxiv.org/abs/1602.00763">SORT: Simple Online and Realtime Tracking (الورقة الأصلية)</a></li>
<li><a href="https://arxiv.org/abs/1703.07402">DeepSORT (الورقة الأصلية)</a></li>
<li><a href="https://arxiv.org/abs/2110.06864">ByteTrack (الورقة الأصلية)</a></li>
<li><a href="https://arxiv.org/abs/2206.14651">BoT-SORT (الورقة الأصلية)</a></li>
<li><a href="https://motchallenge.net/">MOTChallenge — المرجع القياسي لتقييم التتبع</a></li>
</ul>

</div>
